# New Version Test

## Fix 0 — Convert Two-Component System to a Vivarium `PartitionedProcess`

### Changes to the Two-Component System process

Fix 0 migrated the Two-Component System from the original standalone process implementation to a Vivarium `PartitionedProcess` interface so that it can operate within the current simulation framework.

The main changes were:

* Change `TwoComponentSystem` to inherit from `PartitionedProcess`.
* Register the process name and default topology using `topology_registry`, with ports for:

  * `bulk`
  * `listeners`
  * `timestep`
* Replace the original process execution interface with the `PartitionedProcess` methods:

  * `ports_schema()` to define the process ports and defaults.
  * `calculate_request()` to calculate the molecules required for the next timestep.
  * `evolve_state()` to apply the calculated molecule-count changes.
* Build molecule indices from the `bulk` store dynamically using `bulk_name_to_idx()` and obtain molecule counts using the shared `counts()` helper.
* Preserve the existing two-component-system ODE calculation while adapting it to the new process interface.
* Maintain a process-local NumPy random state using the configured seed for stochastic handling when molecule allocations need to be adjusted.
* Continue using the BDF ODE solver for the normal simulation calculation, including the existing handling of negative molecule counts and fallback behaviour.
* Preserve the existing stoichiometric/dependency-matrix calculations and the conversion between independent molecule changes and the full set of molecule-count updates.

The new process interface separates **request calculation** from **state evolution**: `calculate_request()` determines the bulk molecules needed and stores the expected changes, while `evolve_state()` applies those changes and re-solves the system if the requested molecules were not fully allocated.

### Result

Fix 0 therefore primarily changed the **process integration/interface**, rather than changing the underlying Two-Component System reaction model. The existing ODE, stoichiometry, dependency mapping, molecule-allocation, and negative-count handling logic were retained while the process was adapted to the `PartitionedProcess` execution model.


## test 2 - use the old configuration and found that the TU list is not matched

```bash 
(vecoli) katha@IT106990:~/dev/vEcoli-colony-sim$ python ecoli/experiments/ecoli_engine_process.py --config configs/colony_baseline_test2.json
/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/serialize.py:286: UserWarning: Searched through serializers to find <vivarium.core.serialize.NumpyFallbackSerializer object at 0x7fa41580a090> for data of type <class 'ecoli.library.schema.MetadataArray'>. This is inefficient.
  warnings.warn(
Traceback (most recent call last):
  File "/home/katha/dev/vEcoli-colony-sim/ecoli/experiments/ecoli_engine_process.py", line 553, in <module>
    run_simulation(config)
  File "/home/katha/dev/vEcoli-colony-sim/ecoli/experiments/ecoli_engine_process.py", line 418, in run_simulation
    agent_composite = agent_composer.generate(agent_config, path=agent_path)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/composer.py", line 409, in generate
    processes = self.generate_processes(config)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/ecoli/experiments/ecoli_engine_process.py", line 217, in generate_processes
    cell_process = EngineProcess(cell_process_config)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/ecoli/processes/engine_process.py", line 306, in __init__
    self.sim = Engine(
               ^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/engine.py", line 468, in __init__
    self.run_steps()
  File "/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/engine.py", line 833, in run_steps
    update, store = self._calculate_update(
                    ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/engine.py", line 719, in _calculate_update
    return _process_update(
           ^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/engine.py", line 1159, in _process_update
    process = _invoke_process(
              ^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/engine.py", line 1129, in _invoke_process
    process.send_command('next_update', (interval, states))
  File "/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/process.py", line 275, in send_command
    self._command_result = self.run_command_method(
                           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/process.py", line 302, in run_command_method
    return getattr(self, command)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/ecoli/processes/listeners/mass_listener.py", line 232, in next_update
    self.bulk_idx = bulk_name_to_idx(self.bulk_ids, bulk_ids)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/katha/dev/vEcoli-colony-sim/ecoli/library/schema.py", line 343, in bulk_name_to_idx
    raise ValueError(f"Names not found in bulk_names: {missing.tolist()}")
def agent_dirs(history_root):
    return sorted(
        [
            path
            for path in history_root.glob("experiment_id=*__inner/variant=0/lineage_seed=0/generation=*/agent_id=*")
            if path.is_dir()
        ],
        key=lambda path: (
            int(next(part.split("=", 1)[1] for part in path.parts if part.startswith("generation="))),
            next(part.split("=", 1)[1] for part in path.parts if part.startswith("agent_id=")),
        ),
    )

def snapshot_files(history_root):
    return sorted(history_root.rglob("*.pq"), key=lambda path: int(path.stem))

def summarize_run(name, history_root):
    dirs = agent_dirs(history_root)
    files = snapshot_files(history_root)
    stems = sorted({int(path.stem) for path in files})
    print(f\
)
    print(f\
    print(f\
    print(f\
    print(f\
    sample_path = files[-1]
    sample_frame = pd.read_parquet(sample_path, columns=[column for column in ["time", "boundary__length", "boundary__mass", "boundary__width"] if column in pd.read_parquet(sample_path, nrows=0).columns])
    print(f\
    display(sample_frame.head(3).round(4))
    return stems

left_times = summarize_run("baseline_test1", RUNS["baseline_test1"])
right_times = summarize_run("baseline_test3", RUNS["baseline_test3"])
common_times = sorted(set(left_times) & set(right_times))

print(\
)
print(\
print(\
print(\
Exception: ('cell_process',) is not a valid path from ('agents', 'outer')
```

## test 1 - start with empty simdata

```bash 
(vecoli) katha@IT106990:~/dev/vEcoli-colony-sim$ python ecoli/experiments/ecoli_engine_process.py --config configs/colony_baseline_test2.json
```
Results: export error of system final status

## Fix 1 - format the out path of colony final state

### Changes to `ecoli/experiments/ecoli_engine_process.py`

In `colony_save_states()`:

* Detect the parquet emitter and account for the `agents/outer` wrapper.
* Use the correct internal path:

  * Parquet: `agents/outer/agents/<agent_id>/cell_process`
  * Other emitters: `agents/<agent_id>/cell_process`
* Extract `state_to_save["agents"]["outer"]` before saving so that colony JSON files retain the expected `agents/<agent_id>/...` structure.
* Continue extracting each `EngineProcess` inner state with `get_inner_state`.
* Preserve `bulk_dtypes` and `unique_dtypes` metadata.

### Output files

The simulation produces two types of output:

* **Transient simulation state:** emitted continuously using the configured Parquet emitter and saved under the customised `out_dir`, e.g.:

  ```text
  out/colony_runs/baseline_test1/
  ```

* **Final colony state:** saved explicitly by `colony_save_states()` as a JSON file under `/data`, e.g.:

  ```text
  data/baseline_test1_seed_0_colony_t6000.json
  ```

The final JSON is intended to provide a restorable colony state, while the Parquet output captures the transient simulation data.

The fix was verified with a 1-second smoke test. The resulting JSON successfully loads and contains the expected:

```text
agents/
└── 0/
    ├── bulk
    ├── unique
    ├── bulk_dtypes
    └── unique_dtypes
```

rather than the incorrect `agents/outer/...` structure.

## Smoke test

```bash
(vecoli) katha@IT106990:~/dev/vEcoli-colony-sim$ python ecoli/experiments/ecoli_engine_process.py --config /tmp/colony_smoke_test.json

Simulation ID: 2026-08-25_13-13-11_409544+0000
Created: 08/25/2026 at 14:13:16
Description: Test: 2 generations (4 cells), save parquet per cell, save final state
Progress:|██████████████████████████████████████████████████| 0.0/1.0 simulated seconds remaining    
Completed in 2.12 seconds
/home/katha/dev/vEcoli-colony-sim/.venv/lib/python3.12/site-packages/vivarium/core/serialize.py:286: UserWarning: Searched through serializers to find <vivarium.core.serialize.NumpyFallbackSerializer object at 0x7f51683e4560> for data of type <class 'ecoli.library.schema.MetadataArray'>. This is inefficient.
  warnings.warn(
Finished saving the state at t = 1
```

configuration in smoke test:
```text
    ...
   "max_duration": 1,
    "save": true,
    "save_times": [1],
    ...
```

**Notes about the warning**:

The warning says that this works, but Vivarium had to search through the available serializers to find the appropriate fallback.

So:
```text
MetadataArray
    ↓
Vivarium serializer lookup
    ↓
NumpyFallbackSerializer
    ↓
JSON serialization succeeds
```

This is primarily a performance/implementation warning, not a correctness problem.

## test 3 - run with customised emit step

Planned steps:
1. check all file related to the early PR (#390) - all cahnges added (ecoli_engine_process.py, serialize.py)
2. change if needed - skipped
3. set a short test (use colony_baseline_test3.json) and run the simulation
```bash
python ecoli/experiments/ecoli_engine_process.py --config configs/colony_baseline_test3.json
```
4. check results
the file names still the same with test 1, need further checks whether the emit steps are different
    - time step still 1 second.
    - however, no data saved under the subfolder without __inner suffix in test3; there is in test1.

In [ ]:
# Read data format from test1 and test3
import json
from pathlib import Path

PROJECTS = {
    "baseline_test1": Path("/home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test1"),
    "baseline_test3": Path("/home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test3"),
}
FALLBACK_JSON = {
    "baseline_test1": Path("/home/katha/dev/vEcoli-colony-sim/data/baseline_2gen_seed_0_colony_t6000.json"),
    "baseline_test3": Path("/home/katha/dev/vEcoli-colony-sim/data/baseline_test3_seed_10_colony_t6000.json"),
}

def find_json_path(project_name, project_dir):
    json_files = sorted(project_dir.rglob("*.json"))
    if json_files:
        return json_files[-1]

    fallback = FALLBACK_JSON[project_name]
    if fallback.exists():
        print(f"Using fallback JSON for {project_name}: {fallback}")
        return fallback

    raise FileNotFoundError(f"No JSON result found for {project_name} in {project_dir}")

def load_project(project_name, project_dir):
    json_path = find_json_path(project_name, project_dir)
    with open(json_path, "r") as f:
        data = json.load(f)

    agents = data["agents"]
    agent_ids = sorted(agents.keys())
    all_keys = sorted({key for agent in agents.values() for key in agent.keys()})
    return {
        "name": project_name,
        "path": json_path,
        "data": data,
        "agents": agents,
        "agent_ids": agent_ids,
        "all_keys": all_keys,
    }

def print_project_summary(project):
    print(f"\n=== {project['name']} ===")
    print(f"JSON path: {project['path']}")
    print(f"Agent count: {len(project['agent_ids'])}")
    print("Agent IDs:")
    print(project['agent_ids'])
    print("Unique categories (keys):")
    print(project['all_keys'])

    sample_agent = project['agents'][project['agent_ids'][0]]
    print(f"\n--- Data preview for agent '{project['agent_ids'][0]}' ---")
    for key in project['all_keys']:
        value = sample_agent.get(key, None)
        print(f"\nCategory: '{key}'")
        print(f"Type: {type(value)}")
        if isinstance(value, dict):
            print(f"Exclusive subcategories: {sorted(value.keys())}")
        elif isinstance(value, list):
            print(f"Length: {len(value)}")
            print(f"First 3 entries: {value[:3]}")
        else:
            print(f"Value: {value}")

    exchange_keys = set()
    media_ids = set()
    for agent in project['agents'].values():
        env = agent.get('environment', {})
        exchange = env.get('exchange', {})
        exchange_keys.update(exchange.keys())
        media_ids.add(env.get('media_id', None))

    print("\nExclusive list of all exchange chemicals across agents:")
    print(sorted(exchange_keys))
    print("\nExclusive list of all media_id values across agents:")
    print(sorted(media_ids))
    return exchange_keys, media_ids

projects = [load_project(name, path) for name, path in PROJECTS.items()]
project_summaries = {project['name']: print_project_summary(project) for project in projects}

left, right = projects
left_keys = set(left['all_keys'])
right_keys = set(right['all_keys'])
common_keys = sorted(left_keys & right_keys)

print("\n=== Comparison ===")
print("Keys only in baseline_test1:")
print(sorted(left_keys - right_keys))
print("Keys only in baseline_test3:")
print(sorted(right_keys - left_keys))
print("Shared keys:")
print(common_keys)

test1_exchange, test1_media = project_summaries["baseline_test1"]
test3_exchange, test3_media = project_summaries["baseline_test3"]

print("\n=== Environment comparison ===")
print("Exchange chemicals only in baseline_test1:")
print(sorted(test1_exchange - test3_exchange))
print("Exchange chemicals only in baseline_test3:")
print(sorted(test3_exchange - test1_exchange))
print("Media IDs only in baseline_test1:")
print(sorted(test1_media - test3_media))
print("Media IDs only in baseline_test3:")
print(sorted(test3_media - test1_media))

sample_left = left['agents'][left['agent_ids'][0]]
sample_right = right['agents'][right['agent_ids'][0]]
list_diffs = []
for key in common_keys:
    left_value = sample_left.get(key)
    right_value = sample_right.get(key)
    if isinstance(left_value, list) and isinstance(right_value, list):
        left_len = len(left_value)
        right_len = len(right_value)
        if left_len != right_len:
            list_diffs.append((key, left_len, right_len))

print("\n=== First-agent value differences ===")
print("global_time:", sample_left.get("global_time"), sample_right.get("global_time"))
print("timestep:", sample_left.get("timestep"), sample_right.get("timestep"))
print("list-length differences:")
print(list_diffs)

Using fallback JSON for baseline_test1: /home/katha/dev/vEcoli-colony-sim/data/baseline_2gen_seed_0_colony_t6000.json
Using fallback JSON for baseline_test3: /home/katha/dev/vEcoli-colony-sim/data/baseline_test3_seed_10_colony_t6000.json

=== baseline_test1 ===
JSON path: /home/katha/dev/vEcoli-colony-sim/data/baseline_2gen_seed_0_colony_t6000.json
Agent count: 4
Agent IDs:
['000', '001', '010', '011']
Unique categories (keys):
['allocate', 'allocator_rng', 'boundary', 'bulk', 'bulk_dtypes', 'cytoplasm', 'divide', 'division_threshold', 'division_trigger', 'environment', 'global_time', 'listeners', 'next_update_time', 'periplasm', 'process_state', 'request', 'timeline', 'timestep', 'unique', 'unique_dtypes']

--- Data preview for agent '000' ---

Category: 'allocate'
Type: <class 'dict'>
Exclusive subcategories: ['ecoli-chromosome-replication', 'ecoli-complexation', 'ecoli-equilibrium', 'ecoli-polypeptide-elongation', 'ecoli-polypeptide-initiation', 'ecoli-protein-degradation', 'ecoli-r

In [ ]:
# Read data sample from test1 and test3
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

RUNS = {
    "baseline_test1": Path("/home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test1/2026-08-23_20-47-32_975315%2B0000__inner/history"),
    "baseline_test3": Path("/home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test3/2026-08-25_15-22-46_783084%2B0000__inner/history"),
}


def agent_dirs(history_root):
    return sorted(
        [
            path
            for path in history_root.glob("experiment_id=*__inner/variant=0/lineage_seed=0/generation=*/agent_id=*")
            if path.is_dir()
        ],
        key=lambda path: (
            int(next(part.split("=", 1)[1] for part in path.parts if part.startswith("generation="))),
            next(part.split("=", 1)[1] for part in path.parts if part.startswith("agent_id=")),
        ),
    )


def snapshot_files(history_root):
    return sorted(history_root.rglob("*.pq"), key=lambda path: int(path.stem))


def summarize_run(name, history_root):
    dirs = agent_dirs(history_root)
    files = snapshot_files(history_root)
    stems = sorted({int(path.stem) for path in files})

    print(f"\n=== {name} ===")
    print(f"History root: {history_root}")
    print(f"Agent dirs: {len(dirs)}")
    print(f"Parquet snapshots: {len(files)}")
    print(f"Snapshot times: {stems}")

    sample_path = files[-1]
    sample_schema = pq.ParquetFile(sample_path).schema_arrow.names
    sample_columns = [column for column in ["time", "boundary__length", "boundary__mass", "boundary__width"] if column in sample_schema]
    sample_frame = pd.read_parquet(sample_path, columns=sample_columns) if sample_columns else pd.read_parquet(sample_path).head(3)

    print(f"Sample parquet: {sample_path}")
    display(sample_frame.head(3).round(4))
    return stems


left_times = summarize_run("baseline_test1", RUNS["baseline_test1"])
right_times = summarize_run("baseline_test3", RUNS["baseline_test3"])
common_times = sorted(set(left_times) & set(right_times))

print("\n=== Snapshot comparison ===")
print("Only in baseline_test1:", sorted(set(left_times) - set(right_times)))
print("Only in baseline_test3:", sorted(set(right_times) - set(left_times)))
print("Common snapshot times:", common_times)


=== baseline_test1 ===
History root: /home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test1/2026-08-23_20-47-32_975315%2B0000__inner/history
Agent dirs: 7
Parquet snapshots: 29
Snapshot times: [400, 800, 1200, 1600, 2000, 2400, 2529, 2660]
Sample parquet: /home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test1/2026-08-23_20-47-32_975315%2B0000__inner/history/experiment_id=2026-08-23_20-47-32_975315%2B0000__inner/variant=0/lineage_seed=0/generation=2/agent_id=01/2660.pq


,time,boundary__length,boundary__mass,boundary__width
0,4929.0,!units[2.9386553972661176 micrometer],!units[2250.836680478823 femtogram],!units[1.0 micrometer]
1,4930.0,!units[2.9393422699621565 micrometer],!units[2251.4300958881754 femtogram],!units[1.0 micrometer]
2,4931.0,!units[2.940037815687907 micrometer],!units[2252.031004257295 femtogram],!units[1.0 micrometer]



=== baseline_test3 ===
History root: /home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test3/2026-08-25_15-22-46_783084%2B0000__inner/history
Agent dirs: 7
Parquet snapshots: 27
Snapshot times: [400, 800, 1200, 1600, 2000, 2400, 2671, 2800, 2836]
Sample parquet: /home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test3/2026-08-25_15-22-46_783084%2B0000__inner/history/experiment_id=2026-08-25_15-22-46_783084%2B0000__inner/variant=0/lineage_seed=0/generation=2/agent_id=01/2836.pq


,time,boundary__length,boundary__mass,boundary__width
0,5471.0,!units[2.902185899379881 micrometer],!units[2219.329296152958 femtogram],!units[1.0 micrometer]
1,5472.0,!units[2.9027037435469425 micrometer],!units[2219.7766813964677 femtogram],!units[1.0 micrometer]
2,5473.0,!units[2.903230100953602 micrometer],!units[2220.231421550997 femtogram],!units[1.0 micrometer]



=== Snapshot comparison ===
Only in baseline_test1: [2529, 2660]
Only in baseline_test3: [2671, 2800, 2836]
Common snapshot times: [400, 800, 1200, 1600, 2000, 2400]


In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

RUNS = {
    "baseline_test1": Path("/home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test1/2026-08-23_20-47-32_975315%2B0000__inner/history"),
    "baseline_test3": Path("/home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test3/2026-08-25_15-22-46_783084%2B0000__inner/history"),
}


def sample_parquet_path(history_root):
    files = sorted(history_root.rglob("*.pq"), key=lambda path: int(path.stem))
    if not files:
        raise FileNotFoundError(f"No .pq files found under {history_root}")
    return files[-1]


for name, history_root in RUNS.items():
    sample_path = sample_parquet_path(history_root)
    parquet_file = pq.ParquetFile(sample_path)
    schema = parquet_file.schema_arrow
    df = pd.read_parquet(sample_path)

    print(f"\n=== {name} ===")
    print(f"Sample pq file: {sample_path}")
    print("Schema:")
    print(schema)
    print("\nColumns and dtypes:")
    print(df.dtypes)
    print("\nFirst 3 rows:")
    display(df.head(3))


=== baseline_test1 ===
Sample pq file: /home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test1/2026-08-23_20-47-32_975315%2B0000__inner/history/experiment_id=2026-08-23_20-47-32_975315%2B0000__inner/variant=0/lineage_seed=0/generation=2/agent_id=01/2660.pq
Schema:
bulk: large_list<element: int64>
  child 0, element: int64
environment__exchange__4FE-4S: int64
environment__exchange__5-Deoxy-D-Ribofuranose: int64
environment__exchange__ACET: int64
environment__exchange__AMMONIUM: int64
environment__exchange__ARABINOSE: int64
environment__exchange__ARG: int64
environment__exchange__ASN: int64
environment__exchange__BETAINE: int64
environment__exchange__BUTANAL: int64
environment__exchange__CA+2: int64
environment__exchange__CARBON-DIOXIDE: int64
environment__exchange__CARBON-MONOXIDE: int64
environment__exchange__CH33ADO: int64
environment__exchange__CL-: int64
environment__exchange__CO+2: int64
environment__exchange__CPD-10774: int64
environment__exchange__CPD-108: int64
environ

,bulk,environment__exchange__4FE-4S,environment__exchange__5-Deoxy-D-Ribofuranose,environment__exchange__ACET,environment__exchange__AMMONIUM,environment__exchange__ARABINOSE,environment__exchange__ARG,environment__exchange__ASN,environment__exchange__BETAINE,environment__exchange__BUTANAL,...,boundary__length,boundary__outer_surface_area,boundary__inner_surface_area,boundary__mmol_to_counts,boundary__mass,boundary__location,boundary__angle,boundary__thrust,boundary__torque,time
0,"[0, 0, 203, 0, 0, 0, 0, 0, 0, 0, 0, 107, 1, 0,...",0,462494,0,-3620566979,0,0,0,0,0,...,!units[2.9386553972661176 micrometer],!units[9.232058207483231 micrometer ** 2],!units[7.955946585038178 micrometer ** 2],!units[1232259.5743286018 / millimolar],!units[2250.836680478823 femtogram],"[!units[25.68031388583054 micrometer], !units[...",2.192812,0.0,0.0,4929.0
1,"[0, 0, 203, 0, 0, 0, 0, 0, 0, 0, 0, 107, 1, 0,...",0,462614,0,-3621568326,0,0,0,0,0,...,!units[2.9393422699621565 micrometer],!units[9.234216081699056 micrometer ** 2],!units[7.957806184665098 micrometer ** 2],!units[1232584.449885354 / millimolar],!units[2251.4300958881754 femtogram],"[!units[25.67990380604872 micrometer], !units[...",2.196973,0.0,0.0,4930.0
2,"[0, 0, 203, 0, 0, 0, 0, 0, 0, 0, 0, 107, 1, 0,...",0,462756,0,-3622559563,0,0,0,0,0,...,!units[2.940037815687907 micrometer],!units[9.23640120304131 micrometer ** 2],!units[7.95968926515377 micrometer ** 2],!units[1232913.4275928722 / millimolar],!units[2252.031004257295 femtogram],"[!units[25.679053877746526 micrometer], !units...",2.200529,0.0,0.0,4931.0



=== baseline_test3 ===
Sample pq file: /home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test3/2026-08-25_15-22-46_783084%2B0000__inner/history/experiment_id=2026-08-25_15-22-46_783084%2B0000__inner/variant=0/lineage_seed=0/generation=2/agent_id=01/2836.pq
Schema:
bulk: large_list<element: int64>
  child 0, element: int64
environment__exchange__4FE-4S: int64
environment__exchange__5-Deoxy-D-Ribofuranose: int64
environment__exchange__ACET: int64
environment__exchange__AMMONIUM: int64
environment__exchange__ARABINOSE: int64
environment__exchange__ARG: int64
environment__exchange__ASN: int64
environment__exchange__BETAINE: int64
environment__exchange__BUTANAL: int64
environment__exchange__CA+2: int64
environment__exchange__CARBON-DIOXIDE: int64
environment__exchange__CARBON-MONOXIDE: int64
environment__exchange__CH33ADO: int64
environment__exchange__CL-: int64
environment__exchange__CO+2: int64
environment__exchange__CPD-10774: int64
environment__exchange__CPD-108: int64
environ

,bulk,environment__exchange__4FE-4S,environment__exchange__5-Deoxy-D-Ribofuranose,environment__exchange__ACET,environment__exchange__AMMONIUM,environment__exchange__ARABINOSE,environment__exchange__ARG,environment__exchange__ASN,environment__exchange__BETAINE,environment__exchange__BUTANAL,...,boundary__length,boundary__outer_surface_area,boundary__inner_surface_area,boundary__mmol_to_counts,boundary__mass,boundary__location,boundary__angle,boundary__thrust,boundary__torque,time
0,"[0, 0, 247, 0, 0, 0, 0, 0, 0, 0, 0, 35, 1, 0, ...",0,449983,0,-3556143400,0,0,0,0,0,...,!units[2.902185899379881 micrometer],!units[9.117485900843722 micrometer ** 2],!units[7.857211164261726 micrometer ** 2],!units[1215010.3103840766 / millimolar],!units[2219.329296152958 femtogram],"[!units[25.482413973984873 micrometer], !units...",2.356642,0.0,0.0,5471.0
1,"[0, 0, 247, 0, 0, 0, 0, 0, 0, 0, 0, 35, 1, 0, ...",0,450067,0,-3556925269,0,0,0,0,0,...,!units[2.9027037435469425 micrometer],!units[9.119112756274665 micrometer ** 2],!units[7.858613145772163 micrometer ** 2],!units[1215255.2391941096 / millimolar],!units[2219.7766813964677 femtogram],"[!units[25.483323054264606 micrometer], !units...",2.355036,0.0,0.0,5472.0
2,"[0, 0, 247, 0, 0, 0, 0, 0, 0, 0, 0, 35, 1, 0, ...",0,450157,0,-3557712331,0,0,0,0,0,...,!units[2.903230100953602 micrometer],!units[9.12076635683659 micrometer ** 2],!units[7.86003817553779 micrometer ** 2],!units[1215504.1945777275 / millimolar],!units[2220.231421550997 femtogram],"[!units[25.485715609774534 micrometer], !units...",2.354519,0.0,0.0,5473.0


## Debug emit_step passing

**Debug 1**

Findings so far:

- `configs/colony_baseline_test3.json` does set `"emit_step": 60`, while `configs/colony_baseline_test.json` does not set it at all.
- `ecoli/experiments/ecoli_engine_process.py` forwards that value directly into the outer Vivarium engine with `emit_step=config.get("emit_step", 1)`.
- `ecoli/processes/engine_process.py` does not read `emit_step` itself. It only creates the inner simulation and later creates the inner emitter from `inner_emitter`.
- The run layout is the real difference: `baseline_test1` has parquet files under the outer `history/` tree, while `baseline_test3` has an empty outer `history/` tree and only the `__inner/history/` parquet snapshots.
- The sample `.pq` files are per-snapshot parquet tables whose rows still advance by 1 second inside each file. The snapshot filenames are the emitted checkpoints, so `emit_step` does not change the row-level `time` column inside a snapshot.

Debug process used:

1. Searched the codebase for `emit_step` and found only the outer engine handoff in `ecoli/experiments/ecoli_engine_process.py` plus unrelated lysis defaults.
2. Compared the two config files and confirmed `emit_step` is the only relevant config change besides seed.
3. Checked the run directories directly. `baseline_test1/history/` contains parquet files; `baseline_test3/history/` exists but has no files.
4. Verified both runs still write inner parquet snapshots under `__inner/history/`.

Current hypothesis:

- `emit_step` is being passed into the outer engine correctly.
- The remaining problem is likely in outer emitter finalization / outer history writing for the test3 run, not in the config handoff itself.

Next code to inspect:

- `ecoli/library/parquet_emitter.py` for the emit/finalize path and any condition that could suppress outer `history/` writes.
- `ecoli/experiments/ecoli_engine_process.py` around the outer engine teardown path, especially where `emitter.success = True` and `finalize()` are called.

**Debug 2**

Findings so far:

- `emit_step` is present in `configs/colony_baseline_test3.json` and absent from `configs/colony_baseline_test.json`.
- `ecoli/experiments/ecoli_engine_process.py` forwards it into the outer Vivarium engine with `emit_step=config.get("emit_step", 1)`.
- `ecoli/processes/engine_process.py` does not read `emit_step`; it only creates the inner simulation and the inner emitter.
- `ecoli/library/parquet_emitter.py` writes outer parquet files in batches of 400 emits by default (`batch_size=400`) and only flushes the remainder in `finalize()`.
- The outer history tree matches that behavior: test1 has 15 files under `history/`, while test3 has 0 files under the same outer `history/` tree.
- Both runs still have inner per-timestep parquet snapshots under `__inner/history/`.

What that suggests:

- The 60-second setting is not being dropped at the config-to-engine handoff.
- The `emit_step=60` run only produces about 100 outer emits over 6000 s, so it never reaches the default batch flush threshold of 400 during the run.
- If the outer emitter is **not finalized** cleanly at shutdown, the buffered outer data will never be written, which explains why test3 has no outer `history/` files even though the inner run does.
- This also explains why the first record can look the same: the parquet rows are still snapshot rows, and the files are organized by batch index, not by the 60-second emit cadence itself.

Potential fix:

- Explicitly flush the outer ParquetEmitter at the end of `colony_save_states()` or immediately before returning from `run_simulation()`, instead of relying on the implicit shutdown path.
- The safest version would be to set `engine.emitter.success = True` and call `engine.emitter.finalize()` after the final `engine.update(...)` has completed.
- If the intent is to get a physical file every 60 seconds, then `batch_size` also needs to be reconsidered, because the default `400` batches emits together even when the logical emit cadence is 60 s.

Debug steps completed:

1. Searched for `emit_step` and found only the outer engine handoff plus unrelated lysis defaults.
2. Compared the two colony configs and confirmed `emit_step: 60` is the only relevant config difference besides seed.
3. Checked the run trees directly.
4. Counted outer history files: test1 has 15, test3 has 0.
5. Verified both runs still write per-timestep parquet snapshots under `__inner/history/`.
6. Confirmed the outer ParquetEmitter buffers writes in batches of 400 and only flushes leftovers in `finalize()`.

Next code to inspect:

- The outer shutdown/finalization path that should flush the parquet emitter after `colony_save_states()` / `engine.end()`.
- Whether `colony_save_states()` should explicitly finalize the outer emitter when using parquet output.

## Fix2

"Outer Parquet Emitter Shutdown Flush"

### Changes to `ecoli/experiments/ecoli_engine_process.py`

In the normal shutdown path of `run_simulation()`:

* After `engine.end()`, mark the outer Parquet emitter as successful.
* Explicitly call `engine.emitter.finalize()` so that buffered Parquet output is flushed and written when the simulation ends.
* This ensures that short runs that do not reach the emitter's normal batch threshold still produce the expected outer Parquet history files.

### Verification

The fix was verified by running `baseline_test4`.

The resulting **outer Parquet folder saves data at the configured 60-second `emit_step`**, with only three output points:

```text
0 s
60 s
120 s
```

This confirms that the outer Parquet emitter is correctly emitting at the configured cadence and that its buffered data is successfully written at normal shutdown.

At this stage, **no output is produced in the inner Parquet folder**. The missing inner output was subsequently identified as a separate issue related to the inner `ParquetEmitter` not being finalized on normal shutdown, which was addressed in Fix 3.


## Debug: outer/inner subfolder data emission

```bash
python ecoli/experiments/ecoli_engine_process.py --config configs/colony_baseline_test4.json
```

**Debug Record**

- Compared the outputs directly:
  - `baseline_test4` has outer parquet history files and no inner parquet history files.
  - `baseline_test3` has inner parquet history files and no outer parquet history files.
- Traced the code paths:
  - The outer emitter now flushes on normal shutdown in `run_simulation()`, so short runs can write outer `history/` files even if they never reach the batch threshold.
  - The inner `ParquetEmitter` is only finalized on division or exception inside `EngineProcess.next_update()`.
- Root cause:
  - `baseline_test4` is a short run that does not divide, so the inner emitter never reaches its division-based finalize path and its buffered parquet rows stay unwritten.
- Potential fix:
  - Add a normal-shutdown finalize for inner emitters too, or finalize all active `ParquetEmitter` instances after the colony run completes, so short non-dividing runs write both outer and inner parquet output.

In [ ]:
from pathlib import Path

import pandas as pd

run_root = Path("/home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test4_2/2026-08-27_15-32-42_276247%2B0000__inner")
pq_file = sorted(run_root.glob("**/history/**/*.pq"))[0]
frame = pd.read_parquet(pq_file)

print(pq_file)
print(frame)

/home/katha/dev/vEcoli-colony-sim/out/colony_runs/baseline_test4_2/2026-08-27_15-32-42_276247%2B0000__inner/history/experiment_id=2026-08-27_15-32-42_276247%2B0000__inner/variant=0/lineage_seed=0/generation=1/agent_id=0/2.pq
                                                bulk  \
0  [0, 0, 150, 0, 0, 0, 0, 0, 0, 0, 0, 33, 0, 0, ...   
1  [0, 0, 150, 0, 0, 0, 0, 0, 0, 0, 0, 33, 0, 0, ...   

   environment__exchange__4FE-4S  \
0                              0   
1                              0   

   environment__exchange__5-Deoxy-D-Ribofuranose  environment__exchange__ACET  \
0                                              0                            0   
1                                              0                            0   

   environment__exchange__AMMONIUM  environment__exchange__ARABINOSE  \
0                                0                                 0   
1                        -28733823                                 0   

   environment__exchange__ARG  envir

## Fix3

"Inner Parquet Emitter Finalization and Emit Cadence"

### Changes to `ecoli/experiments/ecoli_engine_process.py`

Fix 3 addressed missing inner Parquet output for short, non-dividing colony simulations.

The root cause was that the inner `ParquetEmitter` was only finalized from the division/exception paths inside `EngineProcess.next_update()`. For a short run that did not divide, buffered inner Parquet rows were therefore never written, while the outer emitter was finalized normally at shutdown.

The fix makes the inner simulation follow the configured emission cadence and explicitly finalizes nested Parquet emitters when the colony simulation exits:

* Pass `emit_step` from the top-level simulation configuration into the outer `EcoliEngineProcess` configuration.
* Pass the same `emit_step` through `EcoliEngineProcess.generate_processes()` into the inner `EngineProcess`.
* Add `finalize_parquet_emitters()`, which recursively traverses nested process dictionaries and calls `finalize()` on every `ParquetEmitter`, while avoiding duplicate finalization of the same emitter.
* After `engine.end()`, mark the outer Parquet emitter as successful and finalize it.
* Then recursively finalize the inner Parquet emitters so buffered data from short, non-dividing runs is flushed to disk.

Relevant code changes are in the `EcoliEngineProcess` configuration and process construction, the new `finalize_parquet_emitters()` helper, and the normal shutdown path in `run_simulation()`.

### Verification

Fix 3 was re-tested with `baseline_test4_2`. The resulting output confirms that **both the outer and inner Parquet emitters now use the configured 60-second `emit_step`**.

For the test run, each output contains exactly three emission points:

```text
0 s
60 s
120 s
```

This confirms that the `emit_step` configuration is being passed through to the inner `EngineProcess` correctly, and that both outer and inner Parquet outputs are being emitted at the same 60-second cadence.
The test therefore verifies both parts of Fix 3: **the inner emitter receives the intended emission cadence, and its buffered output is finalized on normal shutdown.**
